In [ ]:
import numpy as np
import math
import time
import pandas as pd
import torch
from statsmodels.tsa.stattools import acf
# =======================================================
# 1. calc_loss_torch
# =======================================================
def calc_loss_torch(params, X_tensor, k_correction):

    p = torch.abs(params) 
    
    bt1, bt2 = p[0], p[1]
    rho1, rho2 = p[2], p[3]
    sg1, sg2 = p[4], p[5]
    

    mu = 0.03; alpha = 0.15; ga1 = 0.15; ga2 = 0.15
    

    A1 = (mu + ga1) * bt1 * bt2
    A2 = (mu + ga1) * ((mu + alpha) * bt2 + (mu + ga2) * bt1) - bt1 * bt2 * mu
    A3 = (mu + ga1) * (mu + alpha) * (mu + ga2)
    R0 = (bt1 * mu) / ((mu + alpha) * (mu + ga1)) + (bt2 * alpha * mu) / ((mu + ga2) * (mu + alpha) * (mu + ga1))
    

    delta_sq = torch.relu(A2**2 - 4 * A1 * A3 * (1 - R0))
    
    Istar = (-A2 + torch.sqrt(delta_sq)) / (2 * A1)
    Sstar = mu / (mu + alpha + bt1 * Istar)
    Vstar = (alpha * mu) / ((mu + alpha + bt1 * Istar) * (mu + ga2 + bt2 * Istar))
    
    mean_vec = torch.stack([Sstar, Vstar, Istar])
    

    a11 = mu + alpha + bt1 * Istar; a13 = bt1 * Sstar
    a21 = alpha; a22 = mu + ga2 + bt2 * Istar; a23 = bt2 * Vstar
    a31 = bt1 * Istar; a32 = bt2 * Istar; a34 = bt1 * Sstar * Istar; a35 = bt2 * Vstar * Istar
    
    tao1 = a11 + a22
    tao2 = a11 * a22 + a13 * a31 + a23 * a32
    tao3 = a13 * a22 * a31 + a13 * a21 * a32 + a11 * a23 * a32
    
    # Delta 1
    a1_val = rho1**2*(tao1*tao2-tao3)+tao2*(rho1*tao2+tao3)
    a2_val = rho1*tao2+tao3; a3_val = rho1+tao1
    a4_val = rho1*tao1*(rho1+tao1)+tao1*tao2-tao3
    a5_val = 2*(rho1**3+rho1**2*tao1+rho1*tao2+tao3)*(tao1*tao2-tao3) + 1e-10
    

    d1_flat = torch.stack([
        a1_val/a5_val, torch.tensor(0.), -a2_val/a5_val, torch.tensor(0.), torch.tensor(0.),
        torch.tensor(0.), a2_val/a5_val, torch.tensor(0.), -a3_val/a5_val, torch.tensor(0.),
        -a2_val/a5_val, torch.tensor(0.), a3_val/a5_val, torch.tensor(0.), torch.tensor(0.),
        torch.tensor(0.), -a3_val/a5_val, torch.tensor(0.), a4_val/(rho1*tao3*a5_val), torch.tensor(0.),
        torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.)
    ])
    delta1 = d1_flat.view(5, 5)
    
    # F1 Matrix
    A_val = -a13*a22 - a23*a32
    B_val = -a13*a21 - a11*a23 + a23*a31
    C_val = a11*a22 - a22*a31 - a21*a32
    
    f1_flat = torch.stack([
        torch.tensor(0.), torch.tensor(-1.), -a13-a22, A_val, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(0.), -(a21+a23), B_val, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(1.), -(a31-a11-a22), C_val, torch.tensor(0.),
        1/a34, tao1/a34, tao2/a34, tao3/a34, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.)
    ])
    F1 = f1_flat.view(5, 5)
    
    # Sig1
    Sig1 = (a34**2) * (sg1**2) * torch.matmul(torch.matmul(F1, delta1), F1.t())
    
    # Delta 2
    v1 = rho2**2*(tao1*tao2-tao3)+tao2*(rho2*tao2+tao3)
    v2 = rho2*tao2+tao3; v3 = rho2+tao1
    v4 = rho2*tao1*(rho2+tao1)+tao1*tao2-tao3
    Q3 = 2*(rho2**3+rho2**2*tao1+rho2*tao2+tao3)*(tao1*tao2-tao3) + 1e-10
    
    d2_flat = torch.stack([
        v1/Q3, torch.tensor(0.), -v2/Q3, torch.tensor(0.), torch.tensor(0.),
        torch.tensor(0.), v2/Q3, torch.tensor(0.), -v3/Q3, torch.tensor(0.),
        -v2/Q3, torch.tensor(0.), v3/Q3, torch.tensor(0.), torch.tensor(0.),
        torch.tensor(0.), -v3/Q3, torch.tensor(0.), v4/(rho2*tao3*Q3), torch.tensor(0.),
        torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.)
    ])
    delta2 = d2_flat.view(5, 5)
    
    # F2 Matrix
    D_val = -a13*(a32-a22)
    E_val = a13*(a21+a31+((-a11+a22-a32)*(a22-a23-a32))/a13)+(-a22+a32)*(-a11+a22-a23-a32)
    F_val_mat = a11*(a32-a22)
    
    f2_flat = torch.stack([
        torch.tensor(0.), torch.tensor(0.), a13, D_val, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(1.), a11+a23, E_val, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(-1.), a32-a11-a22, F_val_mat, torch.tensor(0.),
        torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.), torch.tensor(0.),
        -1/a35, -tao1/a35, -tao2/a35, -tao3/a35, torch.tensor(0.)
    ])
    F2 = f2_flat.view(5, 5)
    
    # Sig2
    Sig2 = (a35**2) * (sg2**2) * torch.matmul(torch.matmul(F2, delta2), F2.t())
    
    # Sigma Total (3x3)
    Sigma = (Sig1 + Sig2)[0:3, 0:3]
    

    #  Log Likelihood
    dist = torch.distributions.MultivariateNormal(loc=mean_vec, covariance_matrix=Sigma)
    
    #LLH
    raw_llh = dist.log_prob(X_tensor) 
    total_raw_llh = torch.sum(raw_llh)
    
    #k
    loss = -total_raw_llh * k_correction
    return loss

# =======================================================
# 2. SDE 
# =======================================================
def sde_Euler(n, t):
    h = t / n
    sqrt_h = math.sqrt(h)
    
    mu = 0.03; alpha = 0.15; ga1 = 0.15; ga2 = 0.15
    bt1 = 3.2; bt2 = 2.0; rho1 = 0.5; rho2 = 0.5
    sg1 = 0.3; sg2 = 0.46
    
    S = np.zeros(n); V = np.zeros(n); I = np.zeros(n)
    cc1 = np.zeros(n); cc2 = np.zeros(n)
    
    S[0] = 0.35; V[0] = 0.25; I[0] = 0.30
    cc1[0] = math.log(bt1); cc2[0] = math.log(bt2)
    
    dW = np.random.normal(0, 1, (n, 2))
    
    for j in range(1, n):
       
        cc1[j] = cc1[j-1] + rho1 * (math.log(bt1) - cc1[j-1]) * h + sg1 * dW[j, 0] * sqrt_h
        cc2[j] = cc2[j-1] + rho2 * (math.log(bt2) - cc2[j-1]) * h + sg2 * dW[j, 1] * sqrt_h
        
        beta1_t = math.exp(cc1[j-1])
        beta2_t = math.exp(cc2[j-1])
        
        S_prev, V_prev, I_prev = S[j-1], V[j-1], I[j-1]
        
        dS = mu - (mu + alpha)*S_prev - beta1_t*S_prev*I_prev
        dV = alpha*S_prev - beta2_t*V_prev*I_prev - (mu + ga2)*V_prev
        dI = beta1_t*S_prev*I_prev + beta2_t*V_prev*I_prev - (mu + ga1)*I_prev
        
        S[j] = S_prev + dS * h
        V[j] = V_prev + dV * h
        I[j] = I_prev + dI * h
        
        # 边界保护
        if S[j] < 0: S[j] = 0
        if V[j] < 0: V[j] = 0
        if I[j] < 0: I[j] = 0
            
    return np.column_stack((S, V, I))
# =======================================================
# 2. Neff
# =======================================================
def calculate_neff(data):
    N = len(data)
    ess_values = []
    
    for col in range(3):
        x = data[:, col]
        ac = acf(x, nlags=min(2000, N // 2), fft=True)
        
        # ACF < 0.05 
        try:
            cutoff = next(i for i, v in enumerate(ac) if v < 0.05)
        except StopIteration:
            cutoff = len(ac)
            
        rho_sum = np.sum(ac[1:cutoff])
        tau = 1 + 2 * rho_sum
        ess = N / tau
        ess_values.append(ess)
        
    return min(ess_values) 

# =======================================================
# 3. SDE
# =======================================================
def sde_Euler(n, t):
    h = t / n
    sqrt_h = math.sqrt(h)
    mu = 0.03; alpha = 0.15; ga1 = 0.15; ga2 = 0.15
    bt1 = 3.2; bt2 = 2.0; rho1 = 0.5; rho2 = 0.5; sg1 = 0.3; sg2 = 0.46
    
    S = np.zeros(n); V = np.zeros(n); I = np.zeros(n)
    cc1 = np.zeros(n); cc2 = np.zeros(n)
    S[0] = 0.35; V[0] = 0.25; I[0] = 0.30
    cc1[0] = math.log(bt1); cc2[0] = math.log(bt2)
    dW = np.random.normal(0, 1, (n, 2))
    
    for j in range(1, n):
        cc1[j] = cc1[j-1] + rho1 * (math.log(bt1) - cc1[j-1]) * h + sg1 * dW[j, 0] * sqrt_h
        cc2[j] = cc2[j-1] + rho2 * (math.log(bt2) - cc2[j-1]) * h + sg2 * dW[j, 1] * sqrt_h
        beta1_t = math.exp(cc1[j-1]); beta2_t = math.exp(cc2[j-1])
        S_prev, V_prev, I_prev = S[j-1], V[j-1], I[j-1]
        
        dS = mu - (mu + alpha)*S_prev - beta1_t*S_prev*I_prev
        dV = alpha*S_prev - beta2_t*V_prev*I_prev - (mu + ga2)*V_prev
        dI = beta1_t*S_prev*I_prev + beta2_t*V_prev*I_prev - (mu + ga1)*I_prev
        S[j] = S_prev + dS * h; V[j] = V_prev + dV * h; I[j] = I_prev + dI * h
        if S[j] < 0: S[j] = 0; 
        if V[j] < 0: V[j] = 0; 
        if I[j] < 0: I[j] = 0
    return np.column_stack((S, V, I))

# =======================================================
# 4. main function
# =======================================================
def run_full_experiment(seed_offset=0):  
    N_REPS = 200
    K_CORRECTION = 0.0009
    
    # target effective sample size
    target_eff_sizes = [50, 100, 300, 500, 1000]
    
    true_vals = np.array([3.2, 2.0, 0.5, 0.5, 0.3, 0.46])
    param_names = ["bt1", "bt2", "rho1", "rho2", "sg1", "sg2"]
    
    print(f"=== 开始全量实验 (Reps={N_REPS}, Seed Offset={seed_offset}) ===")
    
    for n_eff in target_eff_sizes:
        n_steps_needed = int(n_eff / K_CORRECTION)
        t_max_needed = n_steps_needed * 0.01
        burn_in = int(n_steps_needed * 0.1)
        
        print(f"\n>>> Target N_eff = {n_eff} | Steps = {n_steps_needed}")
        
        records = []
        
        start_time = time.time()
        
        for i in range(N_REPS):
            try:
                current_seed = i + n_eff * 1000 + seed_offset
                np.random.seed(current_seed)
                torch.manual_seed(current_seed)
                
                full_traj = sde_Euler(n_steps_needed, t_max_needed)
                clean_data = full_traj[burn_in:, :]
                
                actual_neff = calculate_neff(clean_data)
                
                X_tensor = torch.tensor(clean_data, dtype=torch.float32)
                
                initial_params = true_vals * np.random.uniform(0.90, 1.10, 6)
                params_optim = torch.tensor(initial_params, requires_grad=True, dtype=torch.float32)
                optimizer = torch.optim.Adagrad([params_optim], lr=0.05)
                
                prev_loss = float('inf')
                for iter in range(2000): 
                    optimizer.zero_grad()
                    loss = calc_loss_torch(params_optim, X_tensor, K_CORRECTION)
                    loss.backward()
                    optimizer.step()
                    if iter > 50 and abs(loss.item() - prev_loss) < 1e-4:
                        break
                    prev_loss = loss.item()
                
                est = torch.abs(params_optim).detach().numpy()
                
                row = [i+1] + list(est) + [actual_neff]
                records.append(row)
                
            except Exception as e:
                print(f"Rep {i} error: {e}")
                continue
            
            if (i + 1) % 20 == 0:
                print(".", end="", flush=True)
        
        print()
        
        cols = ["Rep_ID"] + param_names + ["Actual_Neff"]
        df = pd.DataFrame(records, columns=cols)
        
        filename = f"Experiment_Result_N{n_eff}_Offset{seed_offset}.xlsx"
        df.to_excel(filename, index=False)
        print(f"Saved: {filename}")
        
        # summary
        est_matrix = df[param_names].values
        mean_est = np.mean(est_matrix, axis=0)
        bias = mean_est - true_vals
        sd = np.std(est_matrix, axis=0)
        mse = np.mean((est_matrix - true_vals)**2, axis=0)
        avg_neff = df["Actual_Neff"].mean()
        
        print("-" * 60)
        print(f"Summary for Target N={n_eff} (Avg Neff={avg_neff:.1f})")
        print(f"{'Param':<6} | {'Mean':<8} | {'Bias':<8} | {'SD':<8} | {'MSE':<8}")
        print("-" * 60)
        for idx, name in enumerate(param_names):
            print(f"{name:<6} | {mean_est[idx]:.4f}    | {bias[idx]:.4f}    | {sd[idx]:.4f}    | {mse[idx]:.4f}")
        print("-" * 60)

if __name__ == "__main__":
    run_full_experiment(seed_offset=2026) 